Question: do returning visitors buy more often than first time visitors, and is that gap real or just luck?

Numbers:
First time: 6,689 out of 1,263,811 bought
Returning: 5,030 out of 143,769 bought

Below is a two proportion z test on this gap, plus a 95% interval for the difference.

In [1]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

count = np.array([5030, 6689])
nobs = np.array([143769, 1263811])

z_stat, p_value = proportions_ztest(count, nobs)

ci_low, ci_upp = confint_proportions_2indep(
    count1=count[0], nobs1=nobs[0],
    count2=count[1], nobs2=nobs[1],
    method='wald'
)

rate_returning = count[0] / nobs[0]
rate_first_time = count[1] / nobs[1]

print(f"returning rate: {rate_returning:.4%}")
print(f"first-time rate: {rate_first_time:.4%}")
print(f"difference: {(rate_returning - rate_first_time):.4%}")
print(f"95% CI on difference: ({ci_low:.4%}, {ci_upp:.4%})")
print(f"z-stat: {z_stat:.2f}, p-value: {p_value:.2e}")

returning rate: 3.4987%
first-time rate: 0.5293%
difference: 2.9694%
95% CI on difference: (2.8736%, 3.0652%)
z-stat: 117.41, p-value: 0.00e+00


Returning visitors buy about 3 points more often than first time visitors. The interval does not get close to zero, so this gap is real.

The p value is small mostly because the sample is huge. The size of the gap matters more here.

This is still a raw comparison. Returning visitors picked to come back, so some of this gap could just be that more interested people return more. This shows the gap is real, not why it happens.

## Windowed re-run

The comparison above has a problem. Any event after the first day counts toward the returning label, even events after a purchase, so a visitor who buys and then comes back to check their order ends up counting as returning after the fact, and that inflates the gap.

To fix that, I only look at the first 7 days after a visitor's first appearance to decide whether they count as returning or first time. Returning means active on 2 or more distinct days in that window, and anyone who already bought during that same window gets dropped from the comparison, since I want to measure what happens after the label is set, not before. Conversion only counts from day 8 onward. I also drop visitors whose 7 day window runs past the end of the data since I cannot fully observe them. Query is `sql/05_windowed_segmentation.sql`.

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

count = np.array([360, 794])       # converted: returning, first_time
nobs = np.array([76094, 1270818])  # n: returning, first_time

z_stat, p_value = proportions_ztest(count, nobs)
ci_low, ci_upp = confint_proportions_2indep(
    count1=count[0], nobs1=nobs[0],
    count2=count[1], nobs2=nobs[1],
    method='wald'
)

rate_returning = count[0] / nobs[0]
rate_first_time = count[1] / nobs[1]

print(f"returning rate: {rate_returning:.4%}")
print(f"first-time rate: {rate_first_time:.4%}")
print(f"difference: {(rate_returning - rate_first_time):.4%}")
print(f"relative: returning converts {rate_returning/rate_first_time:.2f}x first-time")
print(f"95% CI on difference: ({ci_low:.4%}, {ci_upp:.4%})")
print(f"z-stat: {z_stat:.2f}, p-value: {p_value:.2e}")

returning rate: 0.4731%
first-time rate: 0.0625%
difference: 0.4106%
relative: returning converts 7.57x first-time
95% CI on difference: (0.3617%, 0.4596%)
z-stat: 37.60, p-value: 1.81e-309


I also checked whether this sample was even big enough to catch a smaller gap, using the same 95% confidence idea as above and asking that a real gap get caught 8 times out of 10.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

power_analysis = NormalIndPower()

h_mde = power_analysis.solve_power(
    nobs1=nobs[0], ratio=nobs[1] / nobs[0], alpha=0.05, power=0.8, alternative='two-sided'
)

phi1 = 2 * np.arcsin(np.sqrt(rate_first_time))
p2_mde = np.sin((phi1 + h_mde) / 2) ** 2
mde_pp = (p2_mde - rate_first_time) * 100

observed_h = proportion_effectsize(rate_returning, rate_first_time)
achieved_power = power_analysis.power(
    effect_size=observed_h, nobs1=nobs[0], ratio=nobs[1] / nobs[0], alpha=0.05, alternative='two-sided'
)

print(f"MDE at 80% power: {mde_pp:.4f}pp")
print(f"observed gap: {(rate_returning - rate_first_time) * 100:.4f}pp")
print(f"achieved power at observed gap: {achieved_power:.6f}")

MDE at 80% power: 0.0289pp
observed gap: 0.4106pp
achieved power at observed gap: 1.000000
